In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score

# 1. Carga de archivos
file_guide = "Golden Stores Sueros R.Sur - KO FY'23.xlsx"
file_model = "06_letras_nse_ventas_tradicional_bepensa_zm_merida.xlsx"

# Cargar Modelo Nuevo (Canal Tradicional ZM Mérida)
xls_model = pd.ExcelFile(file_model)
raw_model = pd.read_excel(
    xls_model, sheet_name='Letras por PDV', header=None
)
df_model = raw_model.iloc[5:].copy()
df_model.columns = raw_model.iloc[4].values

# Limpieza y filtrado de PDVs objetivo
df_model['pos_id'] = pd.to_numeric(df_model['pos_id'], errors='coerce')
df_model = df_model[df_model['pos_id'].notna()].copy()
df_model['venta_media_num'] = pd.to_numeric(
    df_model['Venta media (cajas/mes)'], errors='coerce'
)

df_obj = df_model[
    (df_model['venta_media_num'] > 0)
    & (
        df_model['Letra 1 sin AltScore']
        != 'sin NSE (sin hogares a 300 m)'
    )
].copy()

# 2. Cálculo de Accuracy Global y Kappa
y_base = df_obj['Letras base (INEGI + CP)']
y_final = df_obj['Letras (final)']

acc_total = accuracy_score(y_base, y_final)
kappa_total = cohen_kappa_score(y_base, y_final)

print("=== ACCURACY MODELO NUEVO VS MODELO ANTERIOR (BASE) ===")
print(f"Accuracy Total: {acc_total * 100:.2f}%")
print(f"Cohen's Kappa (κ): {kappa_total:.3f}\n")

# Matriz de Confusión / Transición
cm_total = pd.crosstab(
    y_base,
    y_final,
    rownames=['Anterior (Base)'],
    colnames=['Nuevo (Final)'],
    margins=True,
)
print("Matriz de Transición:")
print(cm_total)

# 3. Accuracy individual Letra 1 y Letra 2
acc_l1 = accuracy_score(
    df_obj['Letra 1 sin AltScore'], df_obj['Letra 1 con AltScore (final)']
)
kappa_l1 = cohen_kappa_score(
    df_obj['Letra 1 sin AltScore'], df_obj['Letra 1 con AltScore (final)']
)

acc_l2 = accuracy_score(
    df_obj['Letra 2 sin Rappi'], df_obj['Letra 2 con Rappi (final)']
)
kappa_l2 = cohen_kappa_score(
    df_obj['Letra 2 sin Rappi'], df_obj['Letra 2 con Rappi (final)']
)

print("\n=== DESGLOSE POR COMPONENTE DE LETRA ===")
print(f"Accuracy Letra 1 (NSE / Entorno): {acc_l1 * 100:.2f}% (κ = {kappa_l1:.3f})")
print(f"Accuracy Letra 2 (Ventas / Desempeño): {acc_l2 * 100:.2f}% (κ = {kappa_l2:.3f})")